## Image download Notebook
#### Notebook for downloading the image data and metadata directly from the ISIC archive.

In [1]:
import csv
import os
import random
import requests

#Here we define a python function (download_isic_images) which searches for images in the ISIC archive. 
def download_isic_images(diagnosis, limit, save_dir, records, pool_multiplier=3):
    os.makedirs(save_dir, exist_ok=True)
    url = "https://api.isic-archive.com/api/v2/images/search/"
    params = {
        "query": f'diagnosis_1:"{diagnosis}" AND image_type:"dermoscopic"',
        "limit": limit * pool_multiplier
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    results = response.json().get("results", [])

    #Shuffles the order of the images and selects the desired amount per class. 
    random.shuffle(results)
    selected_results = results[:limit]

    #Loops over selected_results
    #Downloads the image files and saves the image information.
    for i, item in enumerate(selected_results, 1):
        isic_id = item["isic_id"]
        img_url = item["files"]["full"]["url"]

        img_data = requests.get(img_url)
        img_data.raise_for_status()

        filename = f"{isic_id}_{diagnosis.lower()}_dermoscopic.jpg"
        file_path = os.path.join(save_dir, filename)

        with open(file_path, "wb") as f:
            f.write(img_data.content)

        records.append({
            "filename": filename,
            "file_path": file_path,
            "diagnosis": diagnosis,
            "image_type": "dermoscopic"
        })
        print(f"[{i}/{len(selected_results)}] Laddade ner {file_path}")

all_records = []

#Calls the python function (download_isic_images).
#The images are saved in the map "data"
download_isic_images("Benign", 10, "data", all_records)
download_isic_images("Malignant", 10, "data", all_records)

#Shuffles the line order of the classes in the CSV-file.
random.shuffle(all_records)

#The image information is saved in a table-file "labels.csv". 
csv_path = "data/labels.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as csvfile:
    fieldnames = ["filename", "file_path", "diagnosis", "image_type"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_records)

[1/10] Laddade ner data/ISIC_0000007_benign_dermoscopic.jpg
[2/10] Laddade ner data/ISIC_0000015_benign_dermoscopic.jpg
[3/10] Laddade ner data/ISIC_0000027_benign_dermoscopic.jpg
[4/10] Laddade ner data/ISIC_0000010_benign_dermoscopic.jpg
[5/10] Laddade ner data/ISIC_0000009_benign_dermoscopic.jpg
[6/10] Laddade ner data/ISIC_0000000_benign_dermoscopic.jpg
[7/10] Laddade ner data/ISIC_0000033_benign_dermoscopic.jpg
[8/10] Laddade ner data/ISIC_0000034_benign_dermoscopic.jpg
[9/10] Laddade ner data/ISIC_0000023_benign_dermoscopic.jpg
[10/10] Laddade ner data/ISIC_0000001_benign_dermoscopic.jpg
[1/10] Laddade ner data/ISIC_0000141_malignant_dermoscopic.jpg
[2/10] Laddade ner data/ISIC_0000004_malignant_dermoscopic.jpg
[3/10] Laddade ner data/ISIC_0000022_malignant_dermoscopic.jpg
[4/10] Laddade ner data/ISIC_0000013_malignant_dermoscopic.jpg
[5/10] Laddade ner data/ISIC_0000026_malignant_dermoscopic.jpg
[6/10] Laddade ner data/ISIC_0000002_malignant_dermoscopic.jpg
[7/10] Laddade ner da